In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# CSV dosyasını oku
df = pd.read_csv("../../data/hourly_observations.csv")

df["umbrella_needed"].value_counts()

df[["temperature_c", "humidity_pct", "wind_speed_kmh", "precipitation_mm"]].describe()

 
    
#umbrella_needed için
numeric_features = [
    "temperature_c",
    "feels_like_c",
    "dew_point_c",
    "humidity_pct",
    "pressure_hpa",
    "wind_speed_kmh",
    "wind_direction_deg",
    "wind_gust_kmh",
    "cloud_cover_pct",
    "visibility_km",
    "uv_index",
    "hour_of_day",
    "month",
]

categorical_features = [
    "season",
]

features = numeric_features + categorical_features

X = df[features]
y = df["umbrella_needed"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)#stratify ile daha adil split

#encoding
preprocessor = ColumnTransformer(transformers=[("categorical_encoder", OneHotEncoder(handle_unknown="ignore"), categorical_features)],
    remainder="passthrough"  # numeric aynen geçer
)


model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])


model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


scores = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring="f1"   # accuracy değil!
)

ohe = model.named_steps["preprocessor"].named_transformers_["categorical_encoder"]
encoded_categorical_encoder_names = ohe.get_feature_names_out(categorical_features)

all_features = np.concatenate([encoded_categorical_encoder_names, numeric_features])

importances = model.named_steps["classifier"].feature_importances_

feature_importance = pd.DataFrame({
    "feature": all_features,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feature_importance)



print("CV F1 Scores:", scores)
print("Mean F1:", scores.mean())



              precision    recall  f1-score   support

           0       1.00      0.98      0.99      6927
           1       0.94      0.98      0.96      1761

    accuracy                           0.98      8688
   macro avg       0.97      0.98      0.98      8688
weighted avg       0.98      0.98      0.98      8688

               feature  importance
11     cloud_cover_pct    0.459050
6         humidity_pct    0.201216
5          dew_point_c    0.109176
3        temperature_c    0.103189
4         feels_like_c    0.038981
13            uv_index    0.026829
15               month    0.015237
2        season_winter    0.008354
0        season_spring    0.005960
8       wind_speed_kmh    0.005248
9   wind_direction_deg    0.005243
10       wind_gust_kmh    0.005203
12       visibility_km    0.005068
7         pressure_hpa    0.004928
14         hour_of_day    0.004159
1        season_summer    0.002158
CV F1 Scores: [0.96559378 0.9592511  0.97076023 0.9671675  0.96559378]
Mean F1

In [2]:
import joblib
joblib.dump(model, "../../models/umbrella_model.pkl")

['../../models/umbrella_model.pkl']

In [3]:
print(df["umbrella_needed"].value_counts())
print(df[["temperature_c", "humidity_pct", "wind_speed_kmh", "precipitation_mm"]].describe())

umbrella_needed
0    34635
1     8805
Name: count, dtype: int64
       temperature_c  humidity_pct  wind_speed_kmh  precipitation_mm
count   43440.000000  43440.000000    43440.000000      43440.000000
mean        5.626952     58.773545        7.899744          0.219952
std         9.771983     16.949120        5.921891          0.457040
min       -19.700000     30.000000        0.000000          0.000000
25%        -2.200000     44.700000        2.900000          0.000000
50%         5.400000     59.500000        7.500000          0.000000
75%        13.400000     69.100000       12.100000          0.200000
max        31.900000     95.000000       33.500000          4.200000


In [4]:
print(y.value_counts(normalize=True))

umbrella_needed
0    0.797307
1    0.202693
Name: proportion, dtype: float64


In [5]:
sample ={
  "temperature_c": 18.0,
  "feels_like_c": 17.0,
  "dew_point_c": 12.0,
  "humidity_pct": 70.0,
  "pressure_hpa": 1012.0,
  "wind_speed_kmh": 15.0,
  "wind_direction_deg": 180.0,
  "wind_gust_kmh": 22.0,
  "cloud_cover_pct": 90.0,
  "visibility_km": 8.0,
  "uv_index": 2.0,
  "hour_of_day": 14,
  "month": 4,
  "season": "spring",
}
sample_df = pd.DataFrame([sample])
prediction = model.predict(sample_df)
print(prediction)
pred = int(prediction[0])

if pred == 1:
    print("Take an umbrella")
else:
    print("No umbrella needed")

[1]
Take an umbrella
